In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding
import numpy as np

In [ ]:
raw_datasets = load_dataset("glue", "mrpc")
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

In [ ]:
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)


tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="tf")

In [ ]:
tf_train_dataset = tokenized_datasets["train"].to_tf_dataset(
    columns=["attention_mask", "input_ids", "token_type_ids"],
    label_cols=["labels"],
    shuffle=True,
    collate_fn=data_collator,
    batch_size=8,
)

tf_validation_dataset = tokenized_datasets["validation"].to_tf_dataset(
    columns=["attention_mask", "input_ids", "token_type_ids"],
    label_cols=["labels"],
    shuffle=False,
    collate_fn=data_collator,
    batch_size=8,
)

In [ ]:
from transformers import TFAutoModelForSequenceClassification

model = TFAutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

In [ ]:
from tensorflow.keras.losses import SparseCategoricalCrossentropy

In [ ]:
model.compile(
  optimizer="adam",
  loss=SparseCategoricalCrossentropy(from_logits=True),
  metrics=['accuracy']
)

model.fit(
  tf_train_dataset,
  validation_data= tf_validation_dataset
)

In [ ]:
# from tensorflow.keras.optimizers.schedules import PolynomialDecay

In [ ]:
# Cell 1
import tensorflow as tf
from tensorflow.keras.optimizers.schedules import PolynomialDecay
from tensorflow.keras.optimizers import Adam

In [ ]:
batch_size = 8
num_epochs = 3
num_train_steps = len(tf_train_dataset) * num_epochs

# Define learning rate schedule
lr_scheduler = PolynomialDecay(
    initial_learning_rate=5e-5,
    decay_steps=num_train_steps,
    end_learning_rate=0.0,
    power=1.0
)

# Create optimizer with tf.keras.optimizers.legacy.Adam for TF 2.x compatibility
opt = tf.keras.optimizers.Adam.from_config({"learning_rate":lr_scheduler,"beta_1": 0.9,
    "beta_2": 0.999,
    "epsilon": 1e-7})

In [ ]:
# Cell 2
model = TFAutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
model.compile(
    optimizer='adam',
    loss=loss,
    metrics=["accuracy"]
)

In [ ]:
preds = model.predict(tf_validation_dataset)["logits"]
class_preds = np.argmax(preds, axis=1)
print(preds.shape, class_preds.shape)

In [ ]:
import evaluate # type: ignore

metric = evaluate.load("glue", "mrpc")
metric.compute(predictions=class_preds, references=raw_datasets["validation"]["label"])